In [1]:
# Going for flow agents
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

In [7]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('cicids2017_cleaned.csv')


In [13]:
df.head

<bound method NDFrame.head of          Destination Port  Flow Duration  Total Fwd Packets  \
0                      22        1266342                 41   
1                      22        1319353                 41   
2                      22            160                  1   
3                      22        1303488                 41   
4                   35396             77                  1   
...                   ...            ...                ...   
2520746                53          32215                  4   
2520747                53            324                  2   
2520748             58030             82                  2   
2520749                53        1048635                  6   
2520750                53          94939                  4   

         Total Length of Fwd Packets  Fwd Packet Length Max  \
0                               2664                    456   
1                               2664                    456   
2                       

In [14]:
df.tail()

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
2520746,53,32215,4,112,28,28,28.0,0.00000,76,76,...,-1,3,20,0.0,0,0,0.0,0,0,Normal Traffic
2520747,53,324,2,84,42,42,42.0,0.00000,181,181,...,-1,1,20,0.0,0,0,0.0,0,0,Normal Traffic
2520748,58030,82,2,31,31,0,15.5,21.92031,6,6,...,0,0,32,0.0,0,0,0.0,0,0,Normal Traffic
2520749,53,1048635,6,192,32,32,32.0,0.00000,128,128,...,-1,5,20,0.0,0,0,0.0,0,0,Normal Traffic
2520750,53,94939,4,188,47,47,47.0,0.00000,113,113,...,-1,3,20,0.0,0,0,0.0,0,0,Normal Traffic


In [ ]:
df.columns = df.columns.str.strip()

# Use only normal traffic rows for training
normal = df[df['Attack Type'] == 'Normal Traffic']

# Use the columns that map closest to your 7 features
FEATURES = [
    'Flow Duration',
    'Total Fwd Packets',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Flow IAT Std',   # inter-arrival time std — beaconing signal
]

X_normal = normal[FEATURES].replace([float('inf'), -float('inf')], 0).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_normal)

model = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
model.fit(X_scaled)
print("Isolation Forest trained on", len(X_scaled), "normal flows")

In [ ]:
df.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Length of Fwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
       'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length',
       'Max Packet Length', 'Packet Length Mean', 'Packet Length Std',
       'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count',
       'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes',
       'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_p

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
import numpy as np

# Same feature preparation
X_test = df[FEATURES].replace(
    [float('inf'), -float('inf')], 0
).fillna(0)


# True labels
y_true = (
    df['Attack Type'] != 'Normal Traffic'
).astype(int)


# Scale using SAME scaler (do not fit again)
X_test_scaled = scaler.transform(X_test)

In [ ]:
y_pred_if = model.predict(X_test_scaled)

y_pred = np.where(y_pred_if == -1, 1, 0)

In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=['Normal','Attack']
    )
)

              precision    recall  f1-score   support

      Normal       0.83      0.99      0.90   2095057
      Attack       0.07      0.00      0.01    425694

    accuracy                           0.82   2520751
   macro avg       0.45      0.50      0.46   2520751
weighted avg       0.70      0.82      0.75   2520751



In [ ]:
anomaly_score = -model.decision_function(X_test_scaled)

In [ ]:
auc = roc_auc_score(
    y_true,
    anomaly_score
)

print("AUROC:", auc)

AUROC: 0.729251245927116


In [ ]:
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

[[2074106   20951]
 [ 424073    1621]]
